In [2]:
import pandas as pd
import numpy as np

In [4]:
MAX_DAYS = 60
CANDIDATE_K = 10

In [6]:
def load_ghcn(path):
    df = pd.read_csv(path)

    df["date"] = pd.to_datetime(df["date"])
    df["station_id"] = df["station_id"].astype(str)

    # 单位换算（NOAA：十分之一）
    for col in ["PRCP", "TMAX", "TMIN", "TAVG"]:
        if col in df.columns:
            df[col] = df[col] / 10.0

    return df

In [22]:
def load_and_clean_ghcn(csv_gz_path):
    df = pd.read_csv(csv_gz_path, header=None)

    df.columns = [
        "station_id",
        "date",
        "element",
        "value",
        "mflag",
        "qflag",
        "sflag",
        "obs_time"
    ]

    df["station_id"] = df["station_id"].astype(str)
    df["date"] = pd.to_datetime(df["date"], format="%Y%m%d", errors="coerce")

    # ✅ 关键：把 TMAX / TMIN 也加进来
    df = df[df["element"].isin(["PRCP", "TAVG", "TMAX", "TMIN"])]

    # Q-FLAG：只保留为空
    df = df[df["qflag"].isna()]

    # M-FLAG 规则
    allowed = (
        df["mflag"].isna() |
        ((df["mflag"] == "P") & (df["element"] == "PRCP"))
    )
    df = df[allowed]

    # pivot
    df_wide = (
        df
        .pivot_table(
            index=["station_id", "date"],
            columns="element",
            values="value",
            aggfunc="first"
        )
        .reset_index()
    )

    # 单位转换
    for c in ["TAVG", "TMAX", "TMIN"]:
        if c in df_wide.columns:
            df_wide[c] = df_wide[c] / 10.0

    if "PRCP" in df_wide.columns:
        df_wide["PRCP"] = df_wide["PRCP"] / 10.0

    return df_wide

In [20]:
df_2018 = load_ghcn("2018.csv.gz")
df_2019 = load_ghcn("2019.csv.gz")

df_weather = pd.concat([df_2018, df_2019], ignore_index=True)

weather_index = (
    df_weather
    .set_index(["station_id", "date"])
    .sort_index()
)

print("天气数据总行数：", len(df_weather))

KeyError: 'DATE'

In [12]:
df_main = pd.read_csv("1819_with_10_nearest_stations.csv")

df_main["DATE_COL"] = pd.to_datetime(df_main["DATE_COL"])

Index(['AE000041196', '20180101', 'TMAX', '259', 'Unnamed: 4', 'Unnamed: 5',
       'S', 'Unnamed: 7'],
      dtype='object')
   AE000041196  20180101  TMAX  259 Unnamed: 4  Unnamed: 5  S  Unnamed: 7
0  AE000041196  20180101  TMIN  112        NaN         NaN  S         NaN
1  AE000041196  20180101  TAVG  186          H         NaN  S         NaN
2  AEM00041194  20180101  TMAX  250        NaN         NaN  S         NaN
3  AEM00041194  20180101  PRCP    0        NaN         NaN  S         NaN
4  AEM00041194  20180101  TAVG  209          H         NaN  S         NaN
